In [19]:

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# --- Configurazione dei dataset da processare ---
DATASETS = [
    {
        'name'      : 'heloc_ML',
        'input_file': '../data/processed/heloc_dataset_cleanedML.csv',
        'split_dir' : '../data/processed/split_ML',
        'note'      : 'Dataset ML (colonne numeriche + flag binari)'
    },
    {
        'name'      : 'heloc_DLLM',
        'input_file': '../data/processed/heloc_dataset_cleanedDLLM.csv',
        'split_dir' : '../data/processed/split_DLLM',
        'note'      : 'Dataset DLLM (colonne miste: numeriche + stringhe categoriche)'
    },
]

TARGET_COL   = 'RiskPerformance'
RANDOM_STATE = 42

print('Dataset configurati:')
for d in DATASETS:
    print(f"  [{d['name']}]  {d['note']}")
    print(f"    input  : {d['input_file']}")
    print(f"    output : {d['split_dir']}")


Dataset configurati:
  [heloc_ML]  Dataset ML (colonne numeriche + flag binari)
    input  : ../data/processed/heloc_dataset_cleanedML.csv
    output : ../data/processed/split_ML
  [heloc_DLLM]  Dataset DLLM (colonne miste: numeriche + stringhe categoriche)
    input  : ../data/processed/heloc_dataset_cleanedDLLM.csv
    output : ../data/processed/split_DLLM


In [ ]:
# FUNZIONE DI SPLIT — comune a entrambi i dataset


def split_dataset(cfg, target_col=TARGET_COL, random_state=RANDOM_STATE):
    """
    Carica il dataset indicato da cfg, esegue lo split in tre partizioni
    (40% imputation_train / 30% imputation_test / 30% holdout),
    salva i CSV e restituisce un DataFrame di riepilogo.
    """
    name       = cfg['name']
    input_file = cfg['input_file']
    split_dir  = cfg['split_dir']
    os.makedirs(split_dir, exist_ok=True)

    # Caricamento
    # dtype=str preserva i valori stringa del DLLM (es. 'Never Had Delinquency');
    # per il dataset ML i numerici vengono comunque letti correttamente come str
    # e poi salvati invariati
    df = pd.read_csv(input_file, dtype=str)
    total = len(df)

    print(f'\n{"="*60}')
    print(f'  Dataset: {name}  ({total} campioni, {df.shape[1]} colonne)')
    print(f'  Nota   : {cfg["note"]}')
    print(f'{"="*60}')
    print(f'  Distribuzione target ({target_col}):')
    print(df[target_col].value_counts().to_string(header=False))

    X = df.drop(columns=[target_col])
    y = df[target_col]

    # FASE 1: 40% imputation_train | 60% resto
    X_imp_train, X_temp, y_imp_train, y_temp = train_test_split(
        X, y, test_size=0.60, stratify=y, random_state=random_state
    )

    # FASE 2: 50% imputation_test | 50% holdout  sul blocco restante
    X_imp_test, X_holdout, y_imp_test, y_holdout = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state
    )

    # Ricompongo feature + target
    imp_train_df          = X_imp_train.copy()
    imp_train_df[target_col] = y_imp_train.values

    imp_test_df           = X_imp_test.copy()
    imp_test_df[target_col]  = y_imp_test.values

    holdout_df            = X_holdout.copy()
    holdout_df[target_col]   = y_holdout.values

    # Salvataggio CSV
    imp_train_path = os.path.join(split_dir, f'{name}_imputation_train.csv')
    imp_test_path  = os.path.join(split_dir, f'{name}_imputation_test.csv')
    holdout_path   = os.path.join(split_dir, f'{name}_holdout.csv')

    imp_train_df.to_csv(imp_train_path, index=False)
    imp_test_df.to_csv(imp_test_path,   index=False)
    holdout_df.to_csv(holdout_path,     index=False)

    # Riepilogo distribuzione classi
    print()
    for label, X_set, y_set in [
        ('IMPUTATION TRAIN [40%]', X_imp_train, y_imp_train),
        ('IMPUTATION TEST  [30%]', X_imp_test,  y_imp_test),
        ('HOLDOUT          [30%]', X_holdout,   y_holdout),
    ]:
        counts  = y_set.value_counts()
        pct_tot = len(X_set) / total * 100
        print(f'  ── {label} ── {len(X_set)} campioni ({pct_tot:.1f}%)')
        for cls in sorted(counts.index):
            print(f'       {cls}: {counts[cls]:>5}  ({counts[cls]/len(y_set)*100:.1f}%)')

    print(f'\n  File salvati in: {split_dir}')
    print(f'    → {os.path.basename(imp_train_path)}')
    print(f'    → {os.path.basename(imp_test_path)}')
    print(f'    → {os.path.basename(holdout_path)}')

    # DataFrame riepilogo
    riepilogo = pd.DataFrame({
        'Dataset'       : name,
        'Split'         : ['Imputation Train', 'Imputation Test', 'Hold-out', 'TOTALE'],
        'Ruolo'         : [
            'Addestr. modello imputazione',
            'Valut. imputazione + train ML',
            'Test finale (mai corrotto)',
            '—'
        ],
        'N. campioni'   : [len(imp_train_df), len(imp_test_df), len(holdout_df), total],
        '% sul totale'  : [
            f'{len(imp_train_df)/total*100:.1f}%',
            f'{len(imp_test_df)/total*100:.1f}%',
            f'{len(holdout_df)/total*100:.1f}%',
            '100.0%'
        ],
        'Bad'  : [
            int((imp_train_df[target_col]=='Bad').sum()),
            int((imp_test_df[target_col]=='Bad').sum()),
            int((holdout_df[target_col]=='Bad').sum()),
            int((df[target_col]=='Bad').sum())
        ],
        'Good' : [
            int((imp_train_df[target_col]=='Good').sum()),
            int((imp_test_df[target_col]=='Good').sum()),
            int((holdout_df[target_col]=='Good').sum()),
            int((df[target_col]=='Good').sum())
        ],
    })

    riepilogo_path = os.path.join(split_dir, f'{name}_riepilogo_split.csv')
    riepilogo.to_csv(riepilogo_path, index=False)

    return riepilogo

print('Funzione split_dataset definita.')


Funzione split_dataset definita.


In [ ]:
# ESECUZIONE SPLIT SU ENTRAMBI I DATASET

all_riepilogos = []

for cfg in DATASETS:
    riepilogo = split_dataset(cfg)
    all_riepilogos.append(riepilogo)

print('\n' + '='*60)
print('  Split completato per tutti i dataset.')
print('='*60)



  Dataset: heloc_ML  (9861 campioni, 35 colonne)
  Nota   : Dataset ML (colonne numeriche + flag binari)
  Distribuzione target (RiskPerformance):
Bad     5128
Good    4733

  ── IMPUTATION TRAIN [40%] ── 3944 campioni (40.0%)
       Bad:  2051  (52.0%)
       Good:  1893  (48.0%)
  ── IMPUTATION TEST  [30%] ── 2958 campioni (30.0%)
       Bad:  1538  (52.0%)
       Good:  1420  (48.0%)
  ── HOLDOUT          [30%] ── 2959 campioni (30.0%)
       Bad:  1539  (52.0%)
       Good:  1420  (48.0%)

  File salvati in: ../data/processed/split_ML
    → heloc_ML_imputation_train.csv
    → heloc_ML_imputation_test.csv
    → heloc_ML_holdout.csv

  Dataset: heloc_DLLM  (9861 campioni, 24 colonne)
  Nota   : Dataset DLLM (colonne miste: numeriche + stringhe categoriche)
  Distribuzione target (RiskPerformance):
Bad     5128
Good    4733

  ── IMPUTATION TRAIN [40%] ── 3944 campioni (40.0%)
       Bad:  2051  (52.0%)
       Good:  1893  (48.0%)
  ── IMPUTATION TEST  [30%] ── 2958 campioni (30.0%)


In [ ]:
# RIEPILOGO FINALE UNIFICATO

riepilogo_globale = pd.concat(all_riepilogos, ignore_index=True)

print('RIEPILOGO GLOBALE DEGLI SPLIT')
display(riepilogo_globale)


=== RIEPILOGO GLOBALE DEGLI SPLIT ===


,Dataset,Split,Ruolo,N. campioni,% sul totale,Bad,Good
0,heloc_ML,Imputation Train,Addestr. modello imputazione,3944,40.0%,2051,1893
1,heloc_ML,Imputation Test,Valut. imputazione + train ML,2958,30.0%,1538,1420
2,heloc_ML,Hold-out,Test finale (mai corrotto),2959,30.0%,1539,1420
3,heloc_ML,TOTALE,—,9861,100.0%,5128,4733
4,heloc_DLLM,Imputation Train,Addestr. modello imputazione,3944,40.0%,2051,1893
5,heloc_DLLM,Imputation Test,Valut. imputazione + train ML,2958,30.0%,1538,1420
6,heloc_DLLM,Hold-out,Test finale (mai corrotto),2959,30.0%,1539,1420
7,heloc_DLLM,TOTALE,—,9861,100.0%,5128,4733
